<a href="https://colab.research.google.com/github/postnicov/MICpredictOnline/blob/main/OnlinePredictionMICtest1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MOL to PubChem fingerprints, annotation, and MIC prediction
This Google Colab notebook uploads a ZIP archive of `.mol` files, computes PubChem fingerprints in Python, runs the model1 R prediction workflow, passes the prediction results back to Python, enriches the results with IUPAC names and PubChem fingerprint descriptions, displays the final dataframe, and saves it as an Excel file.


In [29]:
#@title # Cell 1: install (if needed) and import packages
import sys
import subprocess
import importlib.util
from IPython.display import Markdown, display

def ensure_package(pkg_name, pip_name=None):
    if importlib.util.find_spec(pkg_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name or pkg_name], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

ensure_package("pandas", "pandas")
ensure_package("rdkit", "rdkit")
ensure_package("skfp", "scikit-fingerprints")
ensure_package("rpy2", "rpy2")
ensure_package("requests", "requests")
ensure_package("openpyxl", "openpyxl")

import zipfile
import shutil
from pathlib import Path

import pandas as pd
from rdkit import Chem
from rdkit.Chem import MolToSmiles
from google.colab import files
from skfp.fingerprints import PubChemFingerprint

display(Markdown("Packages loaded successfully."))


Packages loaded successfully.

In [30]:
#@title # Cell 2: upload a ZIP archive containing multiple .mol files
from IPython.display import Markdown, display
display(Markdown("**Upload a ZIP archive** containing one or more `.mol` files."))
uploaded = files.upload()

if not uploaded:
    raise RuntimeError("No ZIP file was uploaded. Please run this cell again and choose a ZIP archive.")

zip_name = next(iter(uploaded))
work_dir = Path("mol_zip_input")
extract_dir = Path("mol_extracted")

if work_dir.exists():
    shutil.rmtree(work_dir)
if extract_dir.exists():
    shutil.rmtree(extract_dir)

work_dir.mkdir(parents=True, exist_ok=True)
extract_dir.mkdir(parents=True, exist_ok=True)

zip_path = work_dir / zip_name
with open(zip_path, "wb") as f:
    f.write(uploaded[zip_name])

with zipfile.ZipFile(zip_path, "r") as zf:
    zf.extractall(extract_dir)

mol_files = sorted(extract_dir.rglob("*.mol"))
if not mol_files:
    raise FileNotFoundError("The uploaded ZIP was extracted, but no .mol files were found.")

display(Markdown(f"Loaded **{len(mol_files)}** MOL files from `{zip_name}`."))


**Upload a ZIP archive** containing one or more `.mol` files.

Saving mol_test.zip to mol_test (1).zip


Loaded **4** MOL files from `mol_test (1).zip`.

In [31]:
#@title # Cell 3: convert each MOL file to PubChem 881-bit fingerprints and assemble a dataframe
from pathlib import Path
from IPython.display import Markdown, display

extract_dir = Path("mol_extracted")
if "mol_files" not in globals() or not mol_files:
    mol_files = sorted(extract_dir.rglob("*.mol"))

if not mol_files:
    raise FileNotFoundError("No .mol files found. Run the upload/extract cell first, or ensure files exist under mol_extracted/.")

records = []
failed = []
smiles_list = []
names_list = []

for mol_path in mol_files:
    mol = Chem.MolFromMolFile(str(mol_path), removeHs=False)
    if mol is None:
        failed.append(mol_path.name)
        continue
    smiles = MolToSmiles(mol)
    smiles_list.append(smiles)
    names_list.append(mol_path.stem)

fp_transformer = PubChemFingerprint()
fp_array = fp_transformer.transform(smiles_list)

for name, fp in zip(names_list, fp_array):
    row = {"molecule": name}
    for i, bit in enumerate(fp, start=1):
        row[f"PubchemFP{i}"] = int(bit)
    records.append(row)

fingerprints_df = pd.DataFrame(records).set_index("molecule")
fingerprints_df.index.name = "molecule"
csv_path = "pubchem_881_fingerprints.csv"
fingerprints_df.to_csv(csv_path)

display(Markdown(f"Generated a fingerprint table with **{fingerprints_df.shape[0]}** molecules and **{fingerprints_df.shape[1]}** fingerprint columns."))
display(fingerprints_df.head())
if failed:
    display(Markdown("Some MOL files could not be parsed."))


Generated a fingerprint table with **4** molecules and **881** fingerprint columns.

,PubchemFP1,PubchemFP2,PubchemFP3,PubchemFP4,PubchemFP5,PubchemFP6,PubchemFP7,PubchemFP8,PubchemFP9,PubchemFP10,...,PubchemFP872,PubchemFP873,PubchemFP874,PubchemFP875,PubchemFP876,PubchemFP877,PubchemFP878,PubchemFP879,PubchemFP880,PubchemFP881
molecule,,,,,,,,,,,,,,,,,,,,,
105,1,1,1,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
159,0,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
29,1,1,1,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
30,1,1,1,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0


In [32]:
#@title # Cell 4: display the obtained fingerprint table in Colab and add IUPAC names for uploaded molecules
import pandas as pd
import requests
from IPython.display import display, Markdown
from rdkit import Chem
from rdkit.Chem import MolToSmiles
from rdkit.Chem.rdMolDescriptors import CalcMolFormula

fingerprints_df = pd.read_csv("pubchem_881_fingerprints.csv")
fingerprints_df["molecule"] = fingerprints_df["molecule"].astype(str).str.strip()

try:
    from rdkit.Chem import inchi as rd_inchi
    HAS_INCHI = True
except Exception:
    HAS_INCHI = False

session = requests.Session()
session.headers.update({"User-Agent": "Colab-MOL-IUPAC-resolver/1.0"})


def get_json(url, method="GET", data=None, timeout=30):
    try:
        if method == "POST":
            r = session.post(url, data=data, timeout=timeout)
        else:
            r = session.get(url, timeout=timeout)
        if r.status_code == 200:
            return r.json()
    except Exception:
        pass
    return None


def pubchem_property_from_smiles(smiles, prop="IUPACName"):
    url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/smiles/{requests.utils.quote(smiles, safe='')}/property/{prop}/JSON"
    data = get_json(url)
    if data:
        props = data.get("PropertyTable", {}).get("Properties", [])
        if props:
            return props[0].get(prop, "")
    return ""


def pubchem_property_from_inchi(inchi_text, prop="IUPACName"):
    url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/inchi/{requests.utils.quote(inchi_text, safe='')}/property/{prop}/JSON"
    data = get_json(url)
    if data:
        props = data.get("PropertyTable", {}).get("Properties", [])
        if props:
            return props[0].get(prop, "")
    return ""


def pubchem_cid_from_fast_identity_smiles(smiles):
    url = "https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/fastidentity/smiles/cids/JSON"
    data = get_json(url, method="POST", data={"smiles": smiles, "identity_type": "same_connectivity"})
    if data:
        ids = data.get("IdentifierList", {}).get("CID", [])
        if ids:
            return ids[0]
    return None


def pubchem_cid_from_inchikey(inchikey):
    url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/inchikey/{requests.utils.quote(inchikey, safe='')}/cids/JSON"
    data = get_json(url)
    if data:
        ids = data.get("IdentifierList", {}).get("CID", [])
        if ids:
            return ids[0]
    return None


def pubchem_iupac_from_cid(cid):
    url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/{cid}/property/IUPACName/JSON"
    data = get_json(url)
    if data:
        props = data.get("PropertyTable", {}).get("Properties", [])
        if props:
            return props[0].get("IUPACName", "")
    return ""


def resolve_iupac_from_mol(mol):
    if mol is None:
        return "", "unresolved"

    try:
        smiles = MolToSmiles(mol)
        name = pubchem_property_from_smiles(smiles, "IUPACName")
        if name:
            return name, "smiles_property"
    except Exception:
        pass

    if HAS_INCHI:
        try:
            inchi_text = rd_inchi.MolToInchi(mol)
            if inchi_text:
                name = pubchem_property_from_inchi(inchi_text, "IUPACName")
                if name:
                    return name, "inchi_property"
        except Exception:
            pass

        try:
            inchikey = rd_inchi.MolToInchiKey(mol)
            if inchikey:
                cid = pubchem_cid_from_inchikey(inchikey)
                if cid is not None:
                    name = pubchem_iupac_from_cid(cid)
                    if name:
                        return name, "inchikey_to_cid"
        except Exception:
            pass

    try:
        smiles = MolToSmiles(mol)
        cid = pubchem_cid_from_fast_identity_smiles(smiles)
        if cid is not None:
            name = pubchem_iupac_from_cid(cid)
            if name:
                return name, "fastidentity_smiles"
    except Exception:
        pass

    return "", "unresolved"


name_records = []
for mol_path in mol_files:
    mol = Chem.MolFromMolFile(str(mol_path), removeHs=False)
    mol_name = str(mol_path.stem).strip()
    smiles = MolToSmiles(mol) if mol is not None else ""
    formula = CalcMolFormula(mol) if mol is not None else ""

    if HAS_INCHI and mol is not None:
        try:
            inchi_text = rd_inchi.MolToInchi(mol)
        except Exception:
            inchi_text = ""
        try:
            inchikey = rd_inchi.MolToInchiKey(mol)
        except Exception:
            inchikey = ""
    else:
        inchi_text = ""
        inchikey = ""

    iupac_name, iupac_source = resolve_iupac_from_mol(mol) if mol is not None else ("", "invalid_mol")

    if not iupac_name:
        fallback_name = formula if formula else smiles
    else:
        fallback_name = iupac_name

    name_records.append({
        "molecule": mol_name,
        "SMILES": smiles,
        "Formula": formula,
        "InChI": inchi_text,
        "InChIKey": inchikey,
        "IUPAC_name": iupac_name,
        "Name_for_report": fallback_name,
        "IUPAC_source": iupac_source
    })

names_df = pd.DataFrame(name_records)
names_df["molecule"] = names_df["molecule"].astype(str).str.strip()

fingerprints_with_names_df = fingerprints_df.merge(
    names_df[["molecule", "IUPAC_name", "Name_for_report", "IUPAC_source", "Formula", "SMILES", "InChIKey"]],
    on="molecule",
    how="left"
)

cols = ["molecule", "IUPAC_name", "Name_for_report", "IUPAC_source", "Formula", "SMILES", "InChIKey"] + [
    c for c in fingerprints_with_names_df.columns
    if c not in ["molecule", "IUPAC_name", "Name_for_report", "IUPAC_source", "Formula", "SMILES", "InChIKey"]
]
fingerprints_with_names_df = fingerprints_with_names_df[cols]

fingerprints_with_names_df.to_csv("pubchem_881_fingerprints_with_iupac.csv", index=False)

names_df = names_df[["molecule", "IUPAC_name", "Name_for_report", "IUPAC_source", "Formula", "SMILES", "InChIKey"]]

unresolved = names_df[names_df["IUPAC_name"].fillna("").eq("")]
display(Markdown(
    f"Loaded **{len(fingerprints_with_names_df)}** uploaded molecules. "
    f"Resolved IUPAC names for **{len(names_df) - len(unresolved)}** molecules; "
    f"**{len(unresolved)}** remain unresolved in PubChem and use fallback naming in `Name_for_report`."
))
display(fingerprints_with_names_df.head())
if len(unresolved) > 0:
    display(Markdown("**Unresolved molecules:**"))
    display(unresolved[["molecule", "Formula", "SMILES", "InChIKey", "IUPAC_source"]])
print("CSV written to pubchem_881_fingerprints_with_iupac.csv")

[16:35:48] WARNING: Omitted undefined stereo

[16:35:48] WARNING: Omitted undefined stereo

[16:35:49] WARNING: Charges were rearranged

[16:35:49] WARNING: Charges were rearranged

[16:35:50] WARNING: Charges were rearranged



Loaded **4** uploaded molecules. Resolved IUPAC names for **3** molecules; **1** remain unresolved in PubChem and use fallback naming in `Name_for_report`.

,molecule,IUPAC_name,Name_for_report,IUPAC_source,Formula,SMILES,InChIKey,PubchemFP1,PubchemFP2,PubchemFP3,...,PubchemFP872,PubchemFP873,PubchemFP874,PubchemFP875,PubchemFP876,PubchemFP877,PubchemFP878,PubchemFP879,PubchemFP880,PubchemFP881
0,105,,C19H29N3O2,unresolved,C19H29N3O2,CCNC(=O)NC1CCOC2(CCN(Cc3ccccc3)CC2)C1,XVDBDCMBBOFKKH-UHFFFAOYSA-N,1,1,1,...,0,0,0,0,0,0,0,0,0,0
1,159,"3-nitro-[1,2,4]triazolo[4,3-b][1,2,4]triazine","3-nitro-[1,2,4]triazolo[4,3-b][1,2,4]triazine",smiles_property,C4H2N6O2,O=[N+]([O-])c1nnc2nccnn12,PBGSHIFUTLNXSW-UHFFFAOYSA-N,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,29,N-[2-(1-cyclooctyl-5-methylimidazol-2-yl)ethyl...,N-[2-(1-cyclooctyl-5-methylimidazol-2-yl)ethyl...,smiles_property,C19H26N4O4,Cc1cnc(CCNC(=O)c2ccc([N+](=O)[O-])o2)n1C1CCCCCCC1,OHZYCVWUQCSPJA-UHFFFAOYSA-N,1,1,1,...,0,0,0,0,0,0,0,0,0,0
3,30,N-[2-(2-benzyl-5-methylimidazol-1-yl)ethyl]-5-...,N-[2-(2-benzyl-5-methylimidazol-1-yl)ethyl]-5-...,smiles_property,C18H18N4O4,Cc1cnc(Cc2ccccc2)n1CCNC(=O)c1ccc([N+](=O)[O-])o1,FTHSTBZYFNXEEM-UHFFFAOYSA-N,1,1,1,...,0,0,0,0,0,0,0,0,0,0


**Unresolved molecules:**

,molecule,Formula,SMILES,InChIKey,IUPAC_source
0,105,C19H29N3O2,CCNC(=O)NC1CCOC2(CCN(Cc3ccccc3)CC2)C1,XVDBDCMBBOFKKH-UHFFFAOYSA-N,unresolved


CSV written to pubchem_881_fingerprints_with_iupac.csv


In [33]:
#@title Transition to the R-code

%load_ext rpy2.ipython

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


In [34]:
#@title # Cell 6: R prediction workflow using the generated fingerprint table and model1 resources

%%R
needed <- c("dplyr", "writexl")
to_install <- needed[!sapply(needed, requireNamespace, quietly = TRUE)]
if (length(to_install) > 0) {
  suppressWarnings(suppressMessages(
    install.packages(to_install, repos = "https://cloud.r-project.org", quiet = TRUE)
  ))
}

suppressPackageStartupMessages(library(dplyr, warn.conflicts = FALSE))
suppressPackageStartupMessages(library(writexl))

urls <- c(
  features = "https://github.com/postnicov/MICpredictOnline/raw/refs/heads/main/model1/features.rds",
  model = "https://github.com/postnicov/MICpredictOnline/raw/refs/heads/main/model1/model.rds",
  top_features = "https://github.com/postnicov/MICpredictOnline/raw/refs/heads/main/model1/top_features.rds"
)

dest_files <- c(
  features = "features.rds",
  model = "model.rds",
  top_features = "top_features.rds"
)

for (nm in names(urls)) {
  suppressWarnings(suppressMessages(
    download.file(urls[[nm]], destfile = dest_files[[nm]], mode = "wb", quiet = TRUE)
  ))
}

model <- readRDS("model.rds")
expected_feats <- readRDS("features.rds")
top_features <- readRDS("top_features.rds")
new_data <- read.csv("pubchem_881_fingerprints.csv", check.names = FALSE)

if (!("molecule" %in% names(new_data))) {
  stop("The generated fingerprint table must contain a column named 'molecule'.")
}

ids <- new_data$molecule
missing_feats <- setdiff(expected_feats, names(new_data))
for (feat in missing_feats) {
  new_data[[feat]] <- 0
}

new_aligned <- new_data %>%
  select(all_of(expected_feats)) %>%
  mutate(across(everything(), as.numeric))

pred_class <- predict(model, newdata = new_aligned, type = "class")

get_fragments <- function(row_data) {
  active_features <- names(row_data)[as.numeric(row_data) == 1]
  active_features <- intersect(active_features, top_features)
  if (length(active_features) == 0) return(character(0))
  active_features
}

fragment_list <- lapply(seq_len(nrow(new_aligned)), function(i) get_fragments(new_aligned[i, ]))
max_len <- max(c(0, sapply(fragment_list, length)))

fragment_df <- data.frame(
  molecule = ids,
  Predicted_Class = as.character(pred_class),
  stringsAsFactors = FALSE
)

if (max_len > 0) {
  for (j in seq_len(max_len)) {
    fragment_df[[paste0("Associated_Pubchem_", j)]] <- sapply(
      fragment_list,
      function(x) if (length(x) >= j) x[j] else ""
    )
  }
}

write.csv(fragment_df, "r_prediction_results.csv", row.names = FALSE, na = "")
write_xlsx(fragment_df, "blind_prediction_with_fragments.xlsx")


In [35]:
#@title # Cell 7: bring R output back to Python, map fingerprint descriptions, and save the annotated spreadsheet
import pandas as pd
from IPython.display import Markdown, display

r_results_df = pd.read_csv("r_prediction_results.csv")
r_results_df["molecule"] = r_results_df["molecule"].astype(str).str.strip()

names_df = names_df.copy()
names_df["molecule"] = names_df["molecule"].astype(str).str.strip()

final_df = r_results_df.merge(names_df, on="molecule", how="left")

mapping_url = "https://raw.githubusercontent.com/postnicov/MICpredictOnline/refs/heads/main/model1/pubchem_fingerprints_mapping.csv"
mapping_df = pd.read_csv(mapping_url)
mapping_df["Fingerprint"] = mapping_df["Fingerprint"].astype(str).str.strip()
desc_lookup = dict(zip(mapping_df["Fingerprint"], mapping_df["Description"]))

fp_cols = [c for c in final_df.columns if c.startswith("Associated_Pubchem_")]
for col in fp_cols:
    final_df[col] = final_df[col].fillna("").astype(str).str.strip()
    desc_col = col.replace("Associated_Pubchem_", "Description_Pubchem_")
    final_df[desc_col] = final_df[col].map(lambda x: desc_lookup.get(x, "") if x else "")

final_df = final_df.rename(columns={"molecule": "File_Name"})
excel_out = "final_prediction_annotated.xlsx"
final_df.to_excel(excel_out, index=False)

display(Markdown(f"Created the final annotated table for **{len(final_df)}** molecules using the hosted fingerprint mapping file."))
display(final_df.head())
print(f"Saved final annotated table to {excel_out}")

Created the final annotated table for **4** molecules using the hosted fingerprint mapping file.

,File_Name,Predicted_Class,Associated_Pubchem_1,Associated_Pubchem_2,Associated_Pubchem_3,Associated_Pubchem_4,Associated_Pubchem_5,Associated_Pubchem_6,Associated_Pubchem_7,Associated_Pubchem_8,...,Description_Pubchem_6,Description_Pubchem_7,Description_Pubchem_8,Description_Pubchem_9,Description_Pubchem_10,Description_Pubchem_11,Description_Pubchem_12,Description_Pubchem_13,Description_Pubchem_14,Description_Pubchem_15
0,105,2,PubchemFP180,PubchemFP181,PubchemFP345,PubchemFP346,PubchemFP366,PubchemFP639,PubchemFP681,,...,O-C-C-C-N,O-C-C-C-C-C,,,,,,,,
1,159,2,PubchemFP181,PubchemFP345,PubchemFP346,PubchemFP366,PubchemFP396,PubchemFP492,PubchemFP497,PubchemFP540,...,C-N:C-[#1],N:N-C-[#1],N=C-C-[#1],N-C-C-N-C,,,,,,
2,29,1,PubchemFP345,PubchemFP346,PubchemFP358,PubchemFP366,PubchemFP396,PubchemFP492,PubchemFP539,PubchemFP540,...,C-N:C-[#1],N=C-C-C,N=C-C-[#1],C:C-O-C,O-C-C:C-C,C-C:C-O-C,N-C-C-N-C,O-C-C-C-C-C,O=C-C-C-C-C,
3,30,1,PubchemFP180,PubchemFP345,PubchemFP346,PubchemFP358,PubchemFP366,PubchemFP396,PubchemFP492,PubchemFP539,...,N(~C)(~O),C-N:C-[#1],N=C-C-C,N=C-C-[#1],C:C-O-C,O-C-C:C-C,C-C:C-O-C,N-C-C-N-C,O-C-C-C-C-C,O=C-C-C-C-C


Saved final annotated table to final_prediction_annotated.xlsx


In [36]:
#@title # Cell 8: create a multi-page PDF report, one page per molecule
from matplotlib.backends.backend_pdf import PdfPages
import matplotlib.pyplot as plt
from rdkit.Chem import Draw
from IPython.display import Markdown, display
import textwrap

plt.rcParams["font.family"] = "serif"
plt.rcParams["font.size"] = 12

A4_W, A4_H = 8.27, 11.69
LEFT = 2 / 2.54 / A4_W
RIGHT = 1 - (2 / 2.54 / A4_W)
BOTTOM = 2 / 2.54 / A4_H
TOP = 1 - (2 / 2.54 / A4_H)
CONTENT_W = RIGHT - LEFT

HEADER_FS = 12
BODY_FS = 12
TITLE_FS = 12
LINE_H = 0.024


def wrap_text_lines(text, width, break_long_words=True):
    if text is None:
        return [""]
    text = str(text).strip()
    if not text:
        return [""]
    return textwrap.wrap(
        text,
        width=width,
        break_long_words=break_long_words,
        break_on_hyphens=True
    ) or [""]


def draw_wrapped_text(fig, x, y, text, width, fontsize=12, weight="normal", break_long_words=True):
    lines = wrap_text_lines(text, width, break_long_words=break_long_words)
    for line in lines:
        fig.text(
            x, y, line,
            fontsize=fontsize,
            fontfamily="serif",
            fontweight=weight,
            va="top"
        )
        y -= LINE_H
    return y


report_pdf = "molecule_report.pdf"
fp_cols = sorted(
    [c for c in final_df.columns if c.startswith("Associated_Pubchem_")],
    key=lambda x: int(x.split("_")[-1])
)
desc_cols = {c: c.replace("Associated_Pubchem_", "Description_Pubchem_") for c in fp_cols}

with PdfPages(report_pdf) as pdf:
    for _, row in final_df.iterrows():
        mol_name = row["File_Name"]
        mol_path = next((p for p in mol_files if p.stem == mol_name), None)
        mol = Chem.MolFromMolFile(str(mol_path), removeHs=False) if mol_path else None
        img = Draw.MolToImage(mol, size=(1200, 700)) if mol is not None else None

        fig = plt.figure(figsize=(A4_W, A4_H))
        fig.patch.set_facecolor("white")

        y = TOP
        y = draw_wrapped_text(
            fig, LEFT, y,
            f"Molecule file: {mol_name}.mol",
            width=70,
            fontsize=TITLE_FS,
            weight="bold"
        )
        y -= 0.01

        img_h = 0.24
        if img is not None:
            ax_img = fig.add_axes([LEFT, y - img_h, CONTENT_W, img_h])
            ax_img.imshow(img)
            ax_img.axis("off")
            y -= img_h + 0.02
        else:
            y = draw_wrapped_text(fig, LEFT, y, "Structure diagram not available", width=70, fontsize=BODY_FS)
            y -= 0.01

        y = draw_wrapped_text(fig, LEFT, y, "IUPAC substance name", width=70, fontsize=HEADER_FS, weight="bold")
        iupac_text = str(row.get("IUPAC_name", "")).strip()
        formula_text = str(row.get("Formula", "")).strip()
        if iupac_text:
            display_name = iupac_text
        else:
            display_name = f"{formula_text} (name is not resolved)" if formula_text else "Name is not resolved"
        y = draw_wrapped_text(fig, LEFT, y, display_name, width=55, fontsize=BODY_FS, break_long_words=True)
        y -= 0.01

        y = draw_wrapped_text(fig, LEFT, y, "Predicted class", width=70, fontsize=HEADER_FS, weight="bold")
        y = draw_wrapped_text(fig, LEFT, y, row.get("Predicted_Class", ""), width=75, fontsize=BODY_FS)
        y -= 0.01

        y = draw_wrapped_text(fig, LEFT, y, "Detected fingerprints", width=70, fontsize=HEADER_FS, weight="bold")

        entries = []
        for fp_col in fp_cols:
            fp = str(row.get(fp_col, "")).strip()
            desc = str(row.get(desc_cols[fp_col], "")).strip()
            if fp:
                entries.append((fp, desc if desc else "description not found"))

        if not entries:
            y = draw_wrapped_text(fig, LEFT, y, "None", width=75, fontsize=BODY_FS)
        else:
            x_fp = LEFT
            x_desc = LEFT + 0.26

            fig.text(x_fp, y, "Fingerprint", fontsize=HEADER_FS, fontfamily="serif", fontweight="bold", va="top")
            fig.text(x_desc, y, "Description", fontsize=HEADER_FS, fontfamily="serif", fontweight="bold", va="top")
            y -= LINE_H

            fp_wrap = 20
            desc_wrap = 55

            for fp, desc in entries:
                fp_lines = wrap_text_lines(fp, fp_wrap, break_long_words=True)
                desc_lines = wrap_text_lines(desc, desc_wrap, break_long_words=True)
                n_lines = max(len(fp_lines), len(desc_lines))
                needed_h = n_lines * LINE_H

                if y - needed_h < BOTTOM:
                    break

                for i in range(n_lines):
                    fig.text(
                        x_fp, y,
                        fp_lines[i] if i < len(fp_lines) else "",
                        fontsize=BODY_FS,
                        fontfamily="serif",
                        va="top"
                    )
                    fig.text(
                        x_desc, y,
                        desc_lines[i] if i < len(desc_lines) else "",
                        fontsize=BODY_FS,
                        fontfamily="serif",
                        va="top"
                    )
                    y -= LINE_H

        pdf.savefig(fig)
        plt.close(fig)

display(Markdown(
    f"Created the PDF report **{report_pdf}**."
))

Created the PDF report **molecule_report.pdf**.

In [37]:
#@title Set to True when calculating in Colab / Set to False before saving to GitHub

ENABLE_WIDGET = False

In [38]:
#@title # Cell 9: download created files
from pathlib import Path
from google.colab import files
from IPython.display import display, HTML

created_files = [
    "pubchem_881_fingerprints.csv",
    "pubchem_881_fingerprints_with_iupac.csv",
    "blind_prediction_with_fragments.xlsx",
    "molecule_report.pdf"
]

existing_files = [f for f in created_files if Path(f).exists()]

ENABLE_WIDGET = True   # set to False before saving to GitHub

if ENABLE_WIDGET:
    buttons_html = "<h4>Download created files</h4>"
    for f in existing_files:
        buttons_html += f"""
        <button onclick="google.colab.kernel.invokeFunction('download_file', ['{f}'], {{}})"
                style="margin:4px;padding:8px 14px;border:1px solid #999;border-radius:6px;cursor:pointer;">
            Download {f}
        </button><br>
        """
    display(HTML(buttons_html))

    from google.colab import output

    def download_file(filename):
        files.download(filename)

    output.register_callback("download_file", download_file)

else:
    print("Download widget disabled for GitHub-safe save.")
    print("Run this cell in Colab with ENABLE_WIDGET = True to restore download buttons.")
    print("Available files:")
    for f in existing_files:
        print(" -", f)